INTIAL SETUP AND TESTING OF LIBRARIES

In [1]:
%pip install pyspark pyspark[sql] plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName("Spark Practice").master("local[*]").getOrCreate()

In [4]:
df1 = spark.read.csv("day_wise.csv", header=True, inferSchema=True)
df1.show(5)

+----------+---------+------+---------+------+---------+----------+-------------+------------------+---------------------+----------------------+----------------+
|      Date|Confirmed|Deaths|Recovered|Active|New cases|New deaths|New recovered|Deaths / 100 Cases|Recovered / 100 Cases|Deaths / 100 Recovered|No. of countries|
+----------+---------+------+---------+------+---------+----------+-------------+------------------+---------------------+----------------------+----------------+
|2020-01-22|      555|    17|       28|   510|        0|         0|            0|              3.06|                 5.05|                 60.71|               6|
|2020-01-23|      654|    18|       30|   606|       99|         1|            2|              2.75|                 4.59|                  60.0|               8|
|2020-01-24|      941|    26|       36|   879|      287|         8|            6|              2.76|                 3.83|                 72.22|               9|
|2020-01-25|     1434|

Module 1: Data Loading & Schema Handling

Task 1: Load all CSV files into PySpark DataFrames


In [5]:
from pyspark.sql import SparkSession

In [6]:
# Initialize the Spark Session
spark = SparkSession.builder \
    .appName("Covid19_Analysis_Task1") \
    .getOrCreate()

In [7]:
# List of CSV files to be loaded
csv_files = [
    "full_grouped.csv",
    "covid_19_clean_complete.csv",
    "country_wise_latest.csv",
    "day_wise.csv",
    "usa_county_wise.csv",
    "worldometer_data.csv"
]

# Dictionary to hold the DataFrames for subsequent tasks
covid_dataframes = {}

# Process each file according to the requirements
for file in csv_files:
    print(f"\n--- Loading Dataset: {file} ---")
    
    # Load the CSV: Handle headers and infer schema automatically
    df = spark.read.csv(file, header=True, inferSchema=True)
    
    # Store the DataFrame in our dictionary
    # Use the filename (without .csv) as the key
    df_name = file.split(".")[0]
    covid_dataframes[df_name] = df
    
    # Requirement: Print the schema of the dataset
    df.printSchema()
    
    # Requirement: Count and print the number of rows
    print(f"Total Row Count for {file}: {df.count()}")

print("\nTask 1 successfully completed.")


--- Loading Dataset: full_grouped.csv ---
root
 |-- Date: date (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- Confirmed: integer (nullable = true)
 |-- Deaths: integer (nullable = true)
 |-- Recovered: integer (nullable = true)
 |-- Active: integer (nullable = true)
 |-- New cases: integer (nullable = true)
 |-- New deaths: integer (nullable = true)
 |-- New recovered: integer (nullable = true)
 |-- WHO Region: string (nullable = true)

Total Row Count for full_grouped.csv: 35156

--- Loading Dataset: covid_19_clean_complete.csv ---
root
 |-- Province/State: string (nullable = true)
 |-- Country/Region: string (nullable = true)
 |-- Lat: double (nullable = true)
 |-- Long: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Confirmed: integer (nullable = true)
 |-- Deaths: integer (nullable = true)
 |-- Recovered: integer (nullable = true)
 |-- Active: integer (nullable = true)
 |-- WHO Region: string (nullable = true)

Total Row Count for covid_19_cl

Module 2: Data Cleaning Tasks

Task 2: Handle Missing Province/State Values


In [8]:
from pyspark.sql.functions import col

# 1. Select the relevant DataFrame from our dictionary
df_clean = covid_dataframes['covid_19_clean_complete']

# --- Concept: filtering & isNull() ---
# Count nulls per country before filling to create the "Null Count Report"
null_report = df_clean.filter(col("Province/State").isNull()) \
    .groupby("Country/Region") \
    .count() \
    .orderBy("count", ascending=False)

print("--- Country-wise Null Count Report (Province/State) ---")
null_report.show()

# --- Concept: fillna() ---
# Replace null values in the specific column with "Unknown"
df_fixed = df_clean.fillna({"Province/State": "Unknown"})

# --- Verification ---
# Check if any nulls remain in the Province/State column
remaining_nulls = df_fixed.filter(col("Province/State").isNull()).count()
print(f"Remaining Nulls in Province/State: {remaining_nulls}")

# Update our dictionary with the cleaned DataFrame
covid_dataframes['covid_19_clean_complete'] = df_fixed

--- Country-wise Null Count Report (Province/State) ---
+--------------+-----+
|Country/Region|count|
+--------------+-----+
|          Chad|  188|
|      Paraguay|  188|
|        Russia|  188|
|         Yemen|  188|
|       Senegal|  188|
|    Cabo Verde|  188|
|        Sweden|  188|
|        Guyana|  188|
|       Eritrea|  188|
|   Philippines|  188|
|         Burma|  188|
|      Djibouti|  188|
|      Malaysia|  188|
|     Singapore|  188|
|          Fiji|  188|
|        Turkey|  188|
|        Malawi|  188|
|Western Sahara|  188|
|          Iraq|  188|
|       Germany|  188|
+--------------+-----+
only showing top 20 rows
Remaining Nulls in Province/State: 0
